# Experiment 2: sensitivity to futures roll costs

Strategy: the configured DCC-GARCH model with the 60-month calibration rescale (`config/risk_config.json`), primary cost 2 bps, 1976-2014, data conditional on the unverified March 1997 endpoint (`docs/DATA_STATUS.md`). Earlier reports under `reports/*_20261026/` used the rolling-covariance model and are left untouched. Logic lives in `scripts/run_roll_sensitivity.py`; this notebook runs it and reads the output in `reports/dcc_garch_studies/roll/`.

## Goal

The supplied returns are already-rolled continuous series with no contract dates, so actual roll history is unavailable. This experiment charges a hypothetical roll cost on assumed calendars (quarterly Jan/Feb/Mar, or monthly) at 1 or 2 bps, with the 10% band and 1 or 2 bps trade cost. It measures sensitivity, not a forecast of true roll cost.

## Run

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]
REFERENCE = ROOT / "reports/cost_revision_20261026"  # hashed Step 2 inputs (data only)
STUDIES = ROOT / "reports/dcc_garch_studies"
pd.options.display.float_format = "{:.4f}".format
from run_roll_sensitivity import main

OUT = STUDIES / "roll"
if not OUT.exists():
    main(REFERENCE, OUT)

## Data gate

In [2]:
gate = json.loads((OUT / "data_gate.json").read_text())
print("Actual roll history supported:", gate["actual_roll_history_supported"])
print("Missing:", *gate["missing"], sep="\n  - ")

Actual roll history supported: False
Missing:
  - dated delivery-contract identities
  - expiry/actual roll schedules
  - contract multipliers and quote units
  - dated currency conversion conventions
  - vendor continuous-price adjustment lineage
  - execution cost coverage in supplied returns


## Sharpe and cost by calendar (trade cost 2 bps)

In [3]:
runs = pd.read_csv(OUT / "comparison.csv")
view = runs[runs.trade_bps == 2][["calendar", "roll_bps", "strategy", "sharpe", "cagr", "gross_mean_ann", "ordinary_cost_ann", "roll_cost_ann", "roll_turnover_ann"]]
view.sort_values(["strategy", "calendar", "roll_bps"])

,calendar,roll_bps,strategy,sharpe,cagr,gross_mean_ann,ordinary_cost_ann,roll_cost_ann,roll_turnover_ann
33,monthly,1,static,0.4853,0.0433,0.0500,0.0004,0.0023,46.5617
35,monthly,2,static,0.4615,0.0409,0.0500,0.0004,0.0047,46.5718
25,quarterly_feb,1,static,0.5013,0.0450,0.0500,0.0004,0.0008,15.4474
27,quarterly_feb,2,static,0.4933,0.0442,0.0500,0.0004,0.0015,15.4474
21,quarterly_jan,1,static,0.5013,0.0450,0.0500,0.0004,0.0008,15.4684
23,quarterly_jan,2,static,0.4934,0.0442,0.0500,0.0004,0.0015,15.4683
29,quarterly_mar,1,static,0.5011,0.0450,0.0500,0.0004,0.0008,15.6354
31,quarterly_mar,2,static,0.4931,0.0441,0.0500,0.0004,0.0016,15.6353
19,trade_only,0,static,0.5092,0.0458,0.0500,0.0004,0.0000,0.0000
32,monthly,1,trend,1.1738,0.1093,0.1124,0.0017,0.0023,46.8743


## Cost by asset class (monthly calendar, roll 2 bps, TREND)

In [4]:
classes = pd.read_csv(OUT / "class_comparison.csv")
classes.query("run == 'monthly_trade2bps_roll2bps_trend'")[["asset_class", "gross_mean_ann", "ordinary_cost_ann", "roll_cost_ann", "net_mean_ann"]]

,asset_class,gross_mean_ann,ordinary_cost_ann,roll_cost_ann,net_mean_ann
136,COMM,0.0603,0.0007,0.0018,0.0577
137,EQ,0.0136,0.0001,0.0004,0.0132
138,FI,0.0168,0.0004,0.0018,0.0146
139,FX,0.0216,0.0005,0.0007,0.0205


## Interpretation

- Quarterly rolls cost TREND 8-16 bps/yr at 1-2 bps per roll (Sharpe 1.199 to 1.190 and 1.181). The calendar month makes almost no difference.
- Monthly rolls at 2 bps cost 47 bps/yr and cut TREND Sharpe from 1.199 to 1.149. Ordinary trade cost is 17 bps/yr, so roll cost matters only under the extreme monthly calendar.
- STATIC falls by the same cash amount from a lower base (Sharpe 0.509 to 0.462).
- This is a plausibility check on hypothetical calendars. It does not support any claim about actual roll cost, which needs dated contract chains.